# Private Equity in NYC Restaurants
**Story:** New York's independent restaurants are disappearing from the map.

**Data source:** NYS Liquor Authority Active Licenses, April 14 2026

**Analysis by:** Lillian Syme

**Note:** This analysis was developed with assistance from Claude Code (Anthropic). All editorial decisions, source verification, and reporting are the author's own.

In [1]:
import pandas as pd
import os

os.chdir('/Users/lilliansyme/Desktop/Spring 2026/Advanced-Data-Reporting/Private-Equity-Story')

df = pd.read_csv('Data/Current_Liquor_Authority_Active_Licenses_20260414.csv', low_memory=False)
print(f'Total rows: {len(df):,}')
df.head(3)

Total rows: 59,129


,License Permit ID,Premises County,Type,Class,Description,LegalName,DBA,Actual Address Of Premises,Additional Address Information,City,State Name,Zip Code,Original Issue Date,Last Issue Date,Effective Date,Expiration Date,Parent License ID,Legacy Serial Number,AKA Address,Georeference
0,0081-22-231121,Lewis,1,0081,Grocery Store,JOSEPH JOHN STEINER,INDIAN RIVER GENERAL STORE,RR 1 BOX 162 C,NaN,CROGHAN,New York,13327,11/21/2024,11/21/2024,12/01/2024,11/30/2027,NaN,2502674.0,NaN,NaN
1,CM-25-12131,NaN,1,CM,Combined Craft Status,Collage Cellars LLC,NaN,NaN,NaN,NaN,NaN,NaN,06/02/2025,06/02/2025,06/02/2025,05/31/2028,NaN,6061767.0,NaN,NaN
2,0081-23-160415,Westchester,1,0081,Grocery Store,TEJA MART INC,NaN,540 E MAIN ST,NaN,MOUNT KISCO,New York,10549,11/03/2023,11/03/2023,11/03/2023,10/31/2026,NaN,6002777.0,NaN,POINT (-73.72725 41.19272)


### Filter to NYC

In [2]:
nyc_counties = ['New York', 'Kings', 'Queens', 'Bronx', 'Richmond']
nyc = df[df['Premises County'].isin(nyc_counties)].copy()
print(f'NYC licenses: {len(nyc):,}')
nyc['Premises County'].value_counts()

NYC licenses: 24,646


Premises County
New York    9759
Kings       6328
Queens      5375
Bronx       2355
Richmond     829
Name: count, dtype: int64

### Filter to restaurants and bars

In [3]:
nyc_restaurants = nyc[nyc['Description'].str.contains('Restaurant|Bar|Tavern|On-Premise|Catering', case=False, na=False)].copy()
print(f'NYC restaurant/bar licenses: {len(nyc_restaurants):,}')
nyc_restaurants['Description'].value_counts()

NYC restaurant/bar licenses: 11,302


Description
Restaurant                                    8256
Additional Bar                                2142
Catering Establishment                         334
Additional bar- ball park, race track, etc     279
Additional bar- legitimate theater             174
Additional Bar-Seasonal                         51
Cabaret                                         35
Summer Restaurant                               18
Restaurant Brewer                               10
Tavern Miscellaneous                             1
Summer - Tavern Miscellaneous                    1
Winter Restaurant                                1
Name: count, dtype: int64

### Identify investor-backed groups

In [4]:
investor_groups = [
    'MAJOR FOOD GROUP','MOMOFUKU','TAO ','NOBU','CIPRIANI',
    'CATCH HOSPITALITY','EMM GROUP','STARR RESTAURANT','HILLSTONE',
    'LETTUCE ENTERTAIN','RESTAURANT ASSOCIATES','LEVY RESTAURANT',
    'PATINA','BARTACO','ARK RESTAURANT','GENUINE HOSPITALITY',
    'DARDEN','BLOOMIN','BRINKER','SHAKE SHACK','SWEETGREEN',
    'CHIPOTLE','SUBWAY','MCDONALD','BURGER KING','DUNKIN',
    'ARAMARK','LEVY PREMIUM','LEGENDS HOSPITALITY'
]

pattern = '|'.join(investor_groups)
nyc_restaurants['is_group'] = nyc_restaurants['LegalName'].str.upper().str.contains(pattern, na=False)

group_count = nyc_restaurants['is_group'].sum()
total = len(nyc_restaurants)
print(f'Investor/chain-backed licenses: {group_count:,}')
print(f'Independent licenses: {total - group_count:,}')
print(f'Group share: {group_count/total*100:.1f}%')

Investor/chain-backed licenses: 778
Independent licenses: 10,524
Group share: 6.9%


### Top groups by license count

In [5]:
top_groups = (
    nyc_restaurants[nyc_restaurants['is_group']]
    .groupby('LegalName')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
top_groups.columns = ['Legal Name', 'License Count']
top_groups

,Legal Name,License Count
0,LEVY PREMIUM FOODSERVICE LP,187
1,"Aramark Sports and Entertainment Services, LLC",169
2,"LEGENDS HOSPITALITY, LLC & NEW YORK YANKEES PA...",122
3,CHIPOTLE MEXICAN GRILL OF COLORADO LLC,72
4,RESTAURANT ASSOCIATES LLC,57
5,"LEVY PREMIUM FOODSERVICE LP, REST ASSO INC&NY ...",36
6,SHAKE SHACK NEW YORK LLC,25
7,ARAMARK SERVICES INC,20
8,RESTAURANT ASSOCIATES LLC & THOMPSON HOSPITALI...,12
9,"Aramark Educational Services, LLC",9


### Borough breakdown

In [6]:
borough_map = {
    'New York': 'Manhattan',
    'Kings': 'Brooklyn',
    'Queens': 'Queens',
    'Bronx': 'Bronx',
    'Richmond': 'Staten Island'
}
nyc_restaurants['Borough'] = nyc_restaurants['Premises County'].map(borough_map)

borough_summary = nyc_restaurants.groupby('Borough').agg(
    total=('LegalName', 'count'),
    group_backed=('is_group', 'sum')
).reset_index()
borough_summary['pct_group'] = (borough_summary['group_backed'] / borough_summary['total'] * 100).round(1)
borough_summary.sort_values('pct_group', ascending=False)

,Borough,total,group_backed,pct_group
0,Bronx,642,139,21.7
3,Queens,2195,321,14.6
1,Brooklyn,2220,86,3.9
2,Manhattan,5910,227,3.8
4,Staten Island,335,5,1.5


### Export

### Separate venue operators from restaurant investor groups

In [7]:
venue_operators = ['ARAMARK', 'LEVY PREMIUM', 'LEGENDS HOSPITALITY', 'RESTAURANT ASSOCIATES']
venue_pattern = '|'.join(venue_operators)

nyc_restaurants['is_venue_operator'] = nyc_restaurants['LegalName'].str.upper().str.contains(venue_pattern, na=False)
nyc_restaurants['is_pe_restaurant'] = nyc_restaurants['is_group'] & ~nyc_restaurants['is_venue_operator']

venue_count = nyc_restaurants['is_venue_operator'].sum()
pe_count = nyc_restaurants['is_pe_restaurant'].sum()
total = len(nyc_restaurants)

print(f'Venue/stadium operators: {venue_count:,} ({venue_count/total*100:.1f}%)')
print(f'Investor-backed restaurant groups: {pe_count:,} ({pe_count/total*100:.1f}%)')
print(f'Ownership not traceable in public records: {total - venue_count - pe_count:,} ({(total - venue_count - pe_count)/total*100:.1f}%)')

Venue/stadium operators: 651 (5.8%)
Investor-backed restaurant groups: 127 (1.1%)
Ownership not traceable in public records: 10,524 (93.1%)


### Borough breakdown by operator type

In [8]:
borough_summary_split = nyc_restaurants.groupby('Borough').agg(
    total=('LegalName', 'count'),
    venue_operator=('is_venue_operator', 'sum'),
    pe_restaurant=('is_pe_restaurant', 'sum')
).reset_index()
borough_summary_split['pct_venue'] = (borough_summary_split['venue_operator'] / borough_summary_split['total'] * 100).round(1)
borough_summary_split['pct_pe'] = (borough_summary_split['pe_restaurant'] / borough_summary_split['total'] * 100).round(1)
borough_summary_split['pct_total_group'] = ((borough_summary_split['venue_operator'] + borough_summary_split['pe_restaurant']) / borough_summary_split['total'] * 100).round(1)
borough_summary_split.sort_values('pct_total_group', ascending=False)

,Borough,total,venue_operator,pe_restaurant,pct_venue,pct_pe,pct_total_group
0,Bronx,642,126,13,19.6,2.0,21.7
3,Queens,2195,302,19,13.8,0.9,14.6
1,Brooklyn,2220,62,24,2.8,1.1,3.9
2,Manhattan,5910,161,66,2.7,1.1,3.8
4,Staten Island,335,0,5,0.0,1.5,1.5


In [9]:
top_groups.to_csv('Data/top_groups.csv', index=False)
borough_summary.to_csv('Data/borough_breakdown.csv', index=False)
print('Exports saved.')

Exports saved.


### Cross-reference: Confirmed PE-backed groups vs. SLA license data
Searches both LegalName and DBA (doing business as) fields for accuracy.

In [10]:
import pandas as pd

# Each group: search terms matched against both LegalName and DBA columns
pe_groups = {
    'Sant Ambroeus / Felice / Casa Lever': {
        'terms': ['SANT AMBROEUS', 'FELICE', 'CASA LEVER'],
        'investor': 'Three Hills Capital Partners',
        'deal': '$35M growth investment, Sep 2020',
    },
    'Maman': {
        'terms': ['MAMAN'],
        'investor': 'TriSpan',
        'deal': 'Majority stake (~50%), Dec 2020',
    },
    'Dos Toros': {
        'terms': ['DOS TOROS'],
        'investor': 'L Catterton (via Founders Table)',
        'deal': 'Acquisition, Jan 2020',
    },
    'Rosa Mexicano': {
        'terms': ['ROSA MEXICANO'],
        'investor': 'TriSpan',
        'deal': 'Acquisition, Mar 2018',
    },
    'Beauty & Essex': {
        'terms': ['BEAUTY.*ESSEX', 'BEAUTY AND ESSEX'],
        'investor': 'Mohari Hospitality (via Tao Group)',
        'deal': '$550M acquisition, May 2023',
    },
}

results = []
detail_rows = []

for group_name, info in pe_groups.items():
    pattern = '|'.join(info['terms'])
    matches = nyc_restaurants[
        nyc_restaurants['LegalName'].str.upper().str.contains(pattern, na=False, regex=True) |
        nyc_restaurants['DBA'].str.upper().str.contains(pattern, na=False, regex=True)
    ][['LegalName', 'DBA', 'Actual Address Of Premises', 'Premises County']].copy()
    matches['Restaurant Group'] = group_name
    matches['Investor'] = info['investor']
    detail_rows.append(matches)
    results.append({
        'Restaurant Group': group_name,
        'Investor': info['investor'],
        'Deal': info['deal'],
        'NYC Licenses': len(matches),
        'Example LLC Names': ', '.join(matches['LegalName'].unique()[:3]),
    })

summary_df = pd.DataFrame(results).sort_values('NYC Licenses', ascending=False).reset_index(drop=True)
detail_df = pd.concat(detail_rows, ignore_index=True)

print('=== SUMMARY ===')
print(summary_df.to_string())
print(f"\nTotal confirmed PE-backed licenses: {summary_df['NYC Licenses'].sum()}")

=== SUMMARY ===
                      Restaurant Group                            Investor                              Deal  NYC Licenses                                                                       Example LLC Names
0  Sant Ambroeus / Felice / Casa Lever        Three Hills Capital Partners  $35M growth investment, Sep 2020            14                         SA MIDTOWN LLC, 265 LAFAYETTE RISTORANTE LLC, CAFE FOCACCIA INC
1                                Maman                             TriSpan   Majority stake (~50%), Dec 2020            10                                         MAMAN NOMAD LLC, SUGAR BEETS INC, MAMAN UES LLC
2                            Dos Toros    L Catterton (via Founders Table)             Acquisition, Jan 2020             7                              11 CARMINE TACOS LLC, DOS TOROS LLC, DT 1115 Lexington LLC
3                        Rosa Mexicano                             TriSpan             Acquisition, Mar 2018             4  ROSA MEXICANO SE

### Detail: all matching licenses per group

In [11]:
for group in summary_df['Restaurant Group']:
    rows = detail_df[detail_df['Restaurant Group'] == group][['LegalName','DBA','Actual Address Of Premises','Premises County']]
    print(f'\n--- {group} ({len(rows)} licenses) ---')
    print(rows.to_string(index=False))


--- Sant Ambroeus / Felice / Casa Lever (14 licenses) ---
                                    LegalName                                              DBA Actual Address Of Premises Premises County
                               SA MIDTOWN LLC                                       CASA LEVER               390 PARK AVE        New York
                 265 LAFAYETTE RISTORANTE LLC                                    SANT AMBROEUS           265 LAFAYETTE ST        New York
                            CAFE FOCACCIA INC                                  FELICE WINE BAR              400 E 64TH ST        New York
                            FELICE HUDSON LLC                                           FELICE              615 HUDSON ST        New York
                                     SABF LLC                                    SANT AMBROEUS               200 VESEY ST        New York
                          FELICE MONTAGUE LLC                                           FELICE             84 MON

### Export

In [12]:
summary_df.to_csv('Data/pe_license_crossref.csv', index=False)
detail_df.to_csv('Data/pe_license_crossref_detail.csv', index=False)
print('Saved: Data/pe_license_crossref.csv')
print('Saved: Data/pe_license_crossref_detail.csv')

Saved: Data/pe_license_crossref.csv
Saved: Data/pe_license_crossref_detail.csv


### DOH crossref: expanded PE group list vs. all NYC food establishments
Uses NYC Dept of Health inspection data instead of SLA — covers all food establishments regardless of alcohol service. Deduplicates by CAMIS (unique establishment ID).

In [13]:
import pandas as pd

doh = pd.read_csv('Data/DOHMH_Restaurant_Inspections_20260510.csv', low_memory=False)

# One row per unique establishment
doh_unique = doh.drop_duplicates(subset='CAMIS')[['CAMIS','DBA','BORO','BUILDING','STREET','ZIPCODE','CUISINE DESCRIPTION']].copy()
doh_unique['DBA_UPPER'] = doh_unique['DBA'].fillna('').str.upper()
print(f'Unique NYC food establishments: {len(doh_unique):,}')

EXCLUDE_DBAS = {
    'HANAYA OMAKASE INC.',
    'VOSTOCHNAYA KUHNYA',
    'SHASHLICHNAYA/ SEZAM',
    'CHEBURECHNAYA',
}
doh_unique = doh_unique[~doh_unique['DBA'].isin(EXCLUDE_DBAS)].copy()
print(f'After removing known false positives: {len(doh_unique):,}')

pe_groups_expanded = {
    'Sant Ambroeus / Felice / Casa Lever': {
        'terms': ['SANT AMBROEUS', 'FELICE', 'CASA LEVER'],
        'investor': 'Three Hills Capital Partners',
        'deal': '$35M, Sep 2020',
    },
    'Maman': {
        'terms': ['MAMAN'],
        'investor': 'TriSpan',
        'deal': 'Majority stake, Dec 2020',
    },
    'Dos Toros': {
        'terms': ['DOS TOROS'],
        'investor': 'L Catterton (Founders Table)',
        'deal': 'Acquisition, Jan 2020',
    },
    'Rosa Mexicano': {
        'terms': ['ROSA MEXICANO'],
        'investor': 'TriSpan',
        'deal': 'Acquisition, Mar 2018',
    },
    'Tao Group (Tao/Lavo/Marquee/Cathedrale/Beauty & Essex)': {
        'terms': ['TAO DOWNTOWN', 'TAO UPTOWN', 'LAVO NEW YORK', 'CATHEDRALE', 'BEAUTY.*ESSEX', 'BEAUTY AND ESSEX', '^MARQUEE$'],
        'investor': 'Mohari Hospitality',
        'deal': '$550M acquisition, May 2023',
    },
    'Tacombi': {
        'terms': ['TACOMBI'],
        'investor': 'Enlightened Hospitality Investments (EHI)',
        'deal': '$27.5M, Dec 2021',
    },
    'Milk Bar': {
        'terms': ['MILK BAR'],
        'investor': 'RSE Ventures',
        'deal': 'Series A (8-figure), Nov 2017',
    },
    'Bluestone Lane': {
        'terms': ['BLUESTONE LANE'],
        'investor': 'RSE Ventures',
        'deal': 'Growth capital (undisclosed)',
    },
    '&pizza': {
        'terms': ['&PIZZA', 'ANDPIZZA'],
        'investor': 'RSE Ventures',
        'deal': '$25M+, Oct 2017',
    },
    'Dig / Dig Inn': {
        'terms': ['DIG INN', 'DIG FOOD'],
        'investor': 'Enlightened Hospitality Investments (EHI)',
        'deal': '$15M (2019) + $65M (2021)',
    },
    'Joe & the Juice': {
        'terms': ['JOE & THE JUICE', 'JOE AND THE JUICE', 'JOE THE JUICE'],
        'investor': 'General Atlantic (sold to Emirates International Investment, Apr 2026)',
        'deal': '$641M majority acquisition, Nov 2023',
    },
    'Momofuku': {
        'terms': ['MOMOFUKU'],
        'investor': 'RSE Ventures',
        'deal': 'Series A, Dec 2016',
    },
    'Naya': {
        'terms': ['NAYA'],
        'investor': 'TriSpan',
        'deal': 'Majority stake, Oct 2020',
    },
    'Chopt': {
        'terms': ['CHOPT'],
        'investor': 'L Catterton (Founders Table)',
        'deal': 'Acquisition, Jan 2020',
    },
    'Chip City': {
        'terms': ['CHIP CITY'],
        'investor': 'Enlightened Hospitality Investments (EHI)',
        'deal': '$10M Series A Oct 2022 + $7.5M Series B May 2024',
    },
    'Sweet Chick': {
        'terms': ['SWEET CHICK'],
        'investor': 'L Catterton (indirect via Founders Table)',
        'deal': 'Indirect investment, 2022',
    },
    'Fuku': {
        'terms': ['^FUKU$', '^FUKU -'],
        'investor': 'RSE Ventures (via Momofuku)',
        'deal': 'Series A, Dec 2016',
    },
}

results = []
detail_rows = []

for group_name, info in pe_groups_expanded.items():
    pattern = '|'.join(info['terms'])
    matches = doh_unique[doh_unique['DBA_UPPER'].str.contains(pattern, na=False, regex=True)].copy()
    matches['Restaurant Group'] = group_name
    matches['Investor'] = info['investor']
    detail_rows.append(matches)
    results.append({
        'Restaurant Group': group_name,
        'Investor': info['investor'],
        'Deal': info['deal'],
        'NYC Locations (DOH)': len(matches),
        'Example DBAs': ', '.join(matches['DBA'].dropna().unique()[:3]),
    })

doh_summary = pd.DataFrame(results).sort_values('NYC Locations (DOH)', ascending=False).reset_index(drop=True)
doh_detail = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()

print('\n=== DOH CROSSREF SUMMARY ===')
print(doh_summary.to_string())
print(f"\nTotal confirmed PE-backed locations: {doh_summary['NYC Locations (DOH)'].sum()}")

doh_summary.to_csv('Data/doh_pe_crossref.csv', index=False)
doh_detail.to_csv('Data/doh_pe_crossref_detail.csv', index=False)
print('\nSaved: Data/doh_pe_crossref.csv and Data/doh_pe_crossref_detail.csv')


Unique NYC food establishments: 30,967
After removing known false positives: 30,963

=== DOH CROSSREF SUMMARY ===
                                          Restaurant Group                                                                Investor                                              Deal  NYC Locations (DOH)                                                             Example DBAs
0                                          Joe & the Juice  General Atlantic (sold to Emirates International Investment, Apr 2026)              $641M majority acquisition, Nov 2023                   32  JOE & THE JUICE, JOE & THE JUICE BROAD ST, JOE & THE JUICE GREENWICH ST
1                                                     Naya                                                                 TriSpan                          Majority stake, Oct 2020                   29                             NAYA 1177 6TH AVENUE LLC, NAYA, NAYA EXPRESS
2                                                    Maman   


Saved: Data/doh_pe_crossref.csv and Data/doh_pe_crossref_detail.csv


## Additional Analysis: Supporting Story Findings

### Step A: Location growth before vs. after PE deal
Uses earliest DOH inspection date per location as a proxy for opening date.

In [14]:
import pandas as pd

doh = pd.read_csv('Data/DOHMH_Restaurant_Inspections_20260510.csv', low_memory=False)
doh['INSPECTION DATE'] = pd.to_datetime(doh['INSPECTION DATE'], errors='coerce')
doh_full = doh[['CAMIS','DBA','BORO','BUILDING','STREET','ZIPCODE','CUISINE DESCRIPTION','INSPECTION DATE']].copy()
doh_full['DBA_UPPER'] = doh_full['DBA'].fillna('').str.upper()

EXCLUDE_DBAS = {'HANAYA OMAKASE INC.','VOSTOCHNAYA KUHNYA','SHASHLICHNAYA/ SEZAM','CHEBURECHNAYA'}
doh_full = doh_full[~doh_full['DBA'].isin(EXCLUDE_DBAS)].copy()

pe_groups = {
    'Joe & the Juice': {'terms': ['JOE & THE JUICE','JOE AND THE JUICE'], 'investor': 'General Atlantic', 'deal_year': 2023},
    'Naya': {'terms': ['NAYA'], 'investor': 'TriSpan', 'deal_year': 2020},
    'Maman': {'terms': ['MAMAN'], 'investor': 'TriSpan', 'deal_year': 2020},
    'Dos Toros': {'terms': ['DOS TOROS'], 'investor': 'L Catterton', 'deal_year': 2020},
    'Dig / Dig Inn': {'terms': ['DIG INN','DIG FOOD'], 'investor': 'EHI / Danny Meyer', 'deal_year': 2021},
    'Sant Ambroeus / Felice / Casa Lever': {'terms': ['SANT AMBROEUS','FELICE','CASA LEVER'], 'investor': 'Three Hills Capital', 'deal_year': 2020},
    'Bluestone Lane': {'terms': ['BLUESTONE LANE'], 'investor': 'RSE Ventures', 'deal_year': 2019},
    'Tacombi': {'terms': ['TACOMBI'], 'investor': 'EHI / Danny Meyer', 'deal_year': 2021},
    'Milk Bar': {'terms': ['MILK BAR'], 'investor': 'RSE Ventures', 'deal_year': 2017},
    'Momofuku': {'terms': ['MOMOFUKU'], 'investor': 'RSE Ventures', 'deal_year': 2016},
    'Chopt': {'terms': ['CHOPT'], 'investor': 'L Catterton', 'deal_year': 2020},
    'Rosa Mexicano': {'terms': ['ROSA MEXICANO'], 'investor': 'TriSpan', 'deal_year': 2018},
    'Chip City': {'terms': ['CHIP CITY'], 'investor': 'EHI / Danny Meyer', 'deal_year': 2022},
}

detail_rows = []
for group_name, info in pe_groups.items():
    pattern = '|'.join(info['terms'])
    matches = doh_full[doh_full['DBA_UPPER'].str.contains(pattern, na=False, regex=True)].copy()
    matches['Restaurant Group'] = group_name
    matches['Investor'] = info['investor']
    matches['deal_year'] = info['deal_year']
    detail_rows.append(matches)

doh_detail = pd.concat(detail_rows, ignore_index=True)

first_inspect = doh_detail.groupby('CAMIS').agg(
    first_inspection=('INSPECTION DATE', 'min'),
    group=('Restaurant Group', 'first'),
    deal_year=('deal_year', 'first')
).reset_index()
first_inspect['open_year'] = first_inspect['first_inspection'].dt.year
first_inspect['before_deal'] = first_inspect['open_year'] < first_inspect['deal_year']

growth = first_inspect.groupby('group').agg(
    before_deal=('before_deal', 'sum'),
    after_deal=('before_deal', lambda x: (~x).sum()),
    deal_year=('deal_year', 'first')
).reset_index()
growth['total'] = growth['before_deal'] + growth['after_deal']
growth['pct_new_after_deal'] = (growth['after_deal'] / growth['total'] * 100).round(0).astype(int)
growth = growth.sort_values('pct_new_after_deal', ascending=False)
print(growth[['group','deal_year','before_deal','after_deal','total','pct_new_after_deal']].to_string(index=False))
growth.to_csv('Data/growth_after_deal.csv', index=False)
print('\nSaved: Data/growth_after_deal.csv')

                              group  deal_year  before_deal  after_deal  total  pct_new_after_deal
                     Bluestone Lane       2019            0          14     14                 100
                      Dig / Dig Inn       2021            0          20     20                 100
                           Milk Bar       2017            0           7      7                 100
                      Rosa Mexicano       2018            0           3      3                 100
                            Tacombi       2021            0          11     11                 100
                              Maman       2020            1          25     26                  96
Sant Ambroeus / Felice / Casa Lever       2020            1          16     17                  94
                          Dos Toros       2020            2          19     21                  90
                               Naya       2020            3          26     29                  90
          

### Step B: PE location concentration by neighborhood income
Median household income estimates from ACS 2023 5-year estimates, mapped to Manhattan neighborhoods.

In [15]:
neighborhood_income = {
    'Upper East Side': 113000, 'Upper West Side': 107000,
    'Chelsea/Hudson Yards': 102000, 'West Village': 138000,
    'Midtown East': 96000, 'Midtown West': 89000,
    'Flatiron/Gramercy': 104000, 'SoHo/NoLita': 131000,
    'TriBeCa/SoHo': 142000, 'TriBeCa': 142000,
    'Battery Park City': 120000, 'FiDi': 96000,
    'Financial District': 96000, 'Times Sq/Midtown': 89000,
    'East Village/Union Sq': 78000, 'Chelsea': 102000,
    'Lower East Side': 62000, 'Other/Outer Borough': 58000,
}

nb = pd.read_csv('Data/neighborhood_breakdown.csv')
nb['est_median_income'] = nb['Neighborhood'].map(neighborhood_income)
manhattan = nb[nb['Neighborhood'] != 'Other/Outer Borough'].sort_values('PE Locations', ascending=False)
print(manhattan[['Neighborhood','PE Locations','est_median_income']].to_string(index=False))

pe_weighted_income = (manhattan['PE Locations'] * manhattan['est_median_income']).sum() / manhattan['PE Locations'].sum()
print(f'\nWeighted avg median income, Manhattan PE neighborhoods: ${pe_weighted_income:,.0f}')
print(f'Outer borough est. median income: ~$58,000')
print(f'Difference: ${pe_weighted_income - 58000:,.0f}')

         Neighborhood  PE Locations  est_median_income
         Midtown East            24              96000
 Chelsea/Hudson Yards            17             102000
      Upper East Side            17             113000
      Upper West Side            16             107000
         Midtown West            12              89000
         West Village            11             138000
          SoHo/NoLita             9             131000
East Village/Union Sq             8              78000
     Times Sq/Midtown             8              89000
   Financial District             7              96000
    Flatiron/Gramercy             6             104000
    Battery Park City             6             120000
         TriBeCa/SoHo             6             142000
                 FiDi             4              96000
              Chelsea             4             102000
              TriBeCa             2             142000
      Lower East Side             1              62000

Weighted 

### Step C: PE locations in transit hubs (airports, Penn Station, Hudson Yards)

In [16]:
doh_unique = doh_detail.drop_duplicates(subset='CAMIS').copy()
doh_unique['address'] = doh_unique['BUILDING'].fillna('').str.upper() + ' ' + doh_unique['STREET'].fillna('').str.upper()

transit_keywords = ['AIRPORT','PENN STATION','PENN PLAZA','LAGUARDIA','LA GUARDIA','JFK','HUDSON YARDS','TERMINAL','CONCOURSE']
pattern = '|'.join(transit_keywords)
transit = doh_unique[doh_unique['address'].str.contains(pattern, na=False)]

print(f'PE locations in transit hubs: {len(transit)}')
print(transit[['Restaurant Group','DBA','BUILDING','STREET','BORO']].to_string(index=False))

PE locations in transit hubs: 10
Restaurant Group                             DBA  BUILDING                             STREET      BORO
 Joe & the Juice                 JOE & THE JUICE        20                       HUDSON YARDS Manhattan
            Naya                            NAYA        20                       HUDSON YARDS Manhattan
       Dos Toros                       DOS TOROS LAGUARDIA                            AIRPORT    Queens
       Dos Toros DOS TOROS/ZOROS (POST SECURITY)       NaN                         TERMINAL 8    Queens
       Dos Toros                       DOS TOROS        4A         CENTRAL TERMINAL AREA STE4    Queens
       Dos Toros                       DOS TOROS         1                   Central Terminal    Queens
       Dos Toros              DOS TOROS TAQUERIA         1                         PENN PLAZA Manhattan
  Bluestone Lane                  BLUESTONE LANE        30                       HUDSON YARDS Manhattan
  Bluestone Lane               

### Step D: Cuisine diversity -- PE-backed vs. all NYC restaurants

In [17]:
pe_cuisine = doh_unique['CUISINE DESCRIPTION'].value_counts().reset_index()
pe_cuisine.columns = ['Cuisine', 'PE Locations']
pe_cuisine['PE %'] = (pe_cuisine['PE Locations'] / pe_cuisine['PE Locations'].sum() * 100).round(1)

doh_all = doh.drop_duplicates(subset='CAMIS')[['CUISINE DESCRIPTION']].copy()
all_cuisine = doh_all['CUISINE DESCRIPTION'].value_counts().reset_index()
all_cuisine.columns = ['Cuisine', 'All NYC']
all_cuisine['NYC %'] = (all_cuisine['All NYC'] / all_cuisine['All NYC'].sum() * 100).round(1)

merged = pe_cuisine.merge(all_cuisine, on='Cuisine').head(15)
print(merged.to_string(index=False))

                       Cuisine  PE Locations  PE %  All NYC  NYC %
                      American            29  14.9     4910   17.8
Juice, Smoothies, Fruit Salads            21  10.8      547    2.0
                    Coffee/Tea            21  10.8     2216    8.0
                       Mexican            20  10.3     1089    4.0
                Middle Eastern            19   9.8      273    1.0
      Bakery Products/Desserts            15   7.7      923    3.3
                        French            13   6.7      296    1.1
                       Italian            13   6.7     1009    3.7
                       Tex-Mex            10   5.2      310    1.1
                         Other             8   4.1      582    2.1
                 Mediterranean             5   2.6      323    1.2
                    Australian             5   2.6       29    0.1
               Frozen Desserts             2   1.0      379    1.4
            Asian/Asian Fusion             2   1.0      481   

### Step E: Bloomberg M&A deal verification
Cross-references Bloomberg Terminal M&A export against story brands. Source: Bloomberg MA SU GO export, May 2026.

In [18]:
import pandas as pd

bloomberg = pd.read_csv('Data/bloomberg_ma_deals.csv')
bloomberg['Announce Date'] = pd.to_datetime(bloomberg['Announce Date'], errors='coerce')

story_brands = [
    'MAMAN', 'NAYA', 'DOS TOROS', 'TACOMBI', 'BLUESTONE', 'MILK BAR',
    'MOMOFUKU', 'CHIP CITY', 'CHOPT', 'DIG INN', 'ROSA MEXICANO',
    'JOE.*JUICE', 'SANT AMBROEUS', 'FELICE', 'SWEET CHICK', 'FUKU',
    'TAO GROUP', 'BEAUTY.*ESSEX'
]
pattern = '|'.join(story_brands)
story_deals = bloomberg[bloomberg['Target Name'].str.upper().str.contains(pattern, na=False, regex=True)].copy()
story_deals = story_deals.sort_values('Announce Date')

print('=== Bloomberg-confirmed deals for story brands ===')
print(story_deals[['Announce Date','Target Name','Acquirer Name','Announced Total Value (mil.)','Deal Status']].to_string(index=False))

# Flag April 2026 Joe & the Juice sale to Emirates
emirates_deal = bloomberg[
    bloomberg['Target Name'].str.upper().str.contains('JOE.*JUICE', na=False, regex=True) &
    (bloomberg['Announce Date'].dt.year == 2026)
]
if len(emirates_deal) > 0:
    print('\n*** NOTE: Joe & the Juice sold to Emirates International Investment, Apr 2026 ***')
    print(emirates_deal[['Announce Date','Target Name','Acquirer Name','Deal Status']].to_string(index=False))

story_deals.to_csv('Data/bloomberg_story_deals.csv', index=False)
print('\nSaved: Data/bloomberg_story_deals.csv')


=== Bloomberg-confirmed deals for story brands ===
Announce Date                  Target Name                                                                                                                                                                                                  Acquirer Name  Announced Total Value (mil.) Deal Status
   2018-04-04             Rosa Mexicano Co                                                                                                                                                                                        Trispan Rising Stars LP                           NaN   Completed
   2018-07-11  Bluestone Lane Roasting LLC                                                                                                                                                                                               RSE Ventures LLC                           NaN   Completed
   2019-04-10 Dig Inn Restaurant Group LLC                               